In [0]:
pip install azure-eventhub requests-sse

In [0]:
dbutils.library.restartPython()

In [0]:
connection_string = dbutils.secrets.get("pkustra555-scope", "pkustra555-eventhub-cs")

In [0]:
import json
from requests_sse import EventSource
"""
test
"""
url = 'https://stream.wikimedia.org/v2/stream/recentchange'
headers = {"User-Agent": "pkustra555"}
with EventSource(url, headers=headers) as stream:
    for event in stream:
        if event.type == 'message':
            try:
                change = json.loads(event.data)
            except ValueError:
                continue
            else:
                if change['meta']['domain'] == 'canary':
                    continue            
                print('{user} edited {title}'.format(**change))

            print(change)
            break

Azure Event Hub

In [0]:
connection_string = dbutils.secrets.get("pkustra555-scope", "pkustra555-eventhub-cs")

In [0]:
def fetch_wikipedia():
    url = 'https://stream.wikimedia.org/v2/stream/recentchange'
    headers = {"User-Agent": "pkustra555"}
    with EventSource(url, headers=headers) as stream:
        for event in stream:
            if event.type != 'message':
                continue
            try:
                change = json.loads(event.data)
            except ValueError:
                continue

            if change['meta']['domain'] == 'canary':
                continue            
        
            yield change

In [0]:
import json
import time
from datetime import datetime

from azure.eventhub import EventHubProducerClient, EventData
from requests_sse import EventSource

def run(connection_string, n_events = 10):
    producer_client = EventHubProducerClient.from_connection_string(conn_str=connection_string)
    sent_events = 0 
    try:
        with producer_client: 
            for change in fetch_wikipedia():
                event_data_batch = producer_client.create_batch()
                event_json = json.dumps(change, ensure_ascii=False)
                event_data_batch.add(EventData(event_json))

                producer_client.send_batch(event_data_batch)
                sent_events+=1

                print(f"[{datetime.now():%H:%M:%S}] "
                      f"Sent {sent_events} events: "
                      f"{change['user']} - {change['title']}")
                if sent_events >= n_events: 
                    break
    except Exception as e: 
        print(f"Error: {e}")

run(connection_string)